A notebook to find references of each paper from a corpus using Semantic Scholar

## Libraries and Data

In [ ]:
import networkx as nx
import pandas as pd
from tqdm.auto import tqdm
from semanticscholar import SemanticScholar
import time
import os
import re


Data:

In [2]:
# --- Setup ---
DIR = "/home/p0l3/RAD/BERT_PRETRAINING/PRETRAINING/DATASET/ED4RE_2503/ED4RE_2603.pickle"
df = pd.read_pickle(DIR)

# Ensure you have set your API key as an environment variable for higher rate limits
# export SEMANTICSCHOLAR_API_KEY='YourKey'
sch = SemanticScholar()

Helper functions:

In [19]:
# --- Preprocessing ---
def preprocess_doi(doi_string):
    if isinstance(doi_string, list):
        doi_string = doi_string[0]
    if not isinstance(doi_string, str):
        return None
    if doi_string.startswith("/"):
        return doi_string[1:]
    if "https" in doi_string:
        temp_search = re.findall(r"https:\/\/doi\.org(.*)", doi_string)
        if temp_search:
            doi_string = temp_search[0]
        else:
            print(doi_string)
    
    return doi_string.strip()

In [ ]:


# Create a clean list of DOIs from your dataframe
all_dois = [preprocess_doi(doi) for doi in df["DOI"].tolist()]
all_dois = [doi for doi in all_dois if doi] # Remove any None/empty DOIs

print(f"Found {len(all_dois)} valid DOIs to process.")
all_dois = all_dois[::1000]
# --- API Interaction with Best Practices ---
# The fields we need to build the graph
fields = [
    'paperId', 'title', 'externalIds',
    'references.paperId', 'references.externalIds',
    'citations.paperId', 'citations.externalIds'
]

# Batch the DOIs into chunks of 500 (API limit for get_papers)
chunk_size = 10
doi_chunks = [all_dois[i:i + chunk_size] for i in range(0, len(all_dois), chunk_size)]

# Store the API results here
all_s2_papers = []

print(f"Querying Semantic Scholar in {len(doi_chunks)} chunks...")
for chunk in tqdm(doi_chunks, desc="Fetching paper data"):
    try:
        # The corrected API call
        print(chunk)
        results = sch.get_papers(chunk, fields=fields)
        all_s2_papers.extend(results)
    except Exception as e:
        print(f"An error occurred in a chunk: {e}")
    
    # IMPORTANT: Respect the rate limit (100 requests per 5 minutes)
    # A simple sleep after each chunk is a good practice.
    time.sleep(3) # Sleep for 3 seconds to be safe

# --- Graph Construction ---
print("\nBuilding the citation graph...")
G = nx.DiGraph()
edge_list = []

# Create a map from S2 PaperID back to its DOI for convenience
s2id_to_doi = {}
for paper in all_s2_papers:
    if paper.get('externalIds') and paper['externalIds'].get('DOI'):
        s2id_to_doi[paper['paperId']] = paper['externalIds']['DOI']

# Process the results to create nodes and edges
for paper in tqdm(all_s2_papers, desc="Processing results"):
    if not paper or not paper.get('paperId'):
        continue

    source_id = paper['paperId']
    source_doi = s2id_to_doi.get(source_id, source_id) # Use DOI as ID if available
    G.add_node(source_doi, title=paper.get('title', 'No Title'))
    
    # Add edges from this paper's references
    if paper.get('references'):
        for ref in paper['references']:
            if ref and ref.get('paperId'):
                target_id = ref['paperId']
                target_doi = s2id_to_doi.get(target_id, target_id)
                # Add the edge: source -> target (source cites target)
                edge_list.append((source_doi, target_doi))

# Add all edges at once for efficiency
G.add_edges_from(edge_list)

# --- Analysis and Visualization ---
print("\n--- Graph Analysis ---")
print(f"Graph created with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

# Filter out nodes that are not in your original corpus for a cleaner view
nodes_in_original_corpus = set(all_dois)
internal_graph = G.subgraph([node for node in G.nodes() if node in nodes_in_original_corpus])

print(f"Internal graph (only papers from your corpus) has {internal_graph.number_of_nodes()} nodes and {internal_graph.number_of_edges()} edges.")

# You can now proceed with your visualization or further analysis on `G` or `internal_graph`
# For example, find the most cited paper within your corpus
if internal_graph.number_of_nodes() > 0:
    top_cited = sorted(internal_graph.in_degree(), key=lambda x: x[1], reverse=True)[:5]
    print("\nTop 5 most cited papers within your corpus:")
    for node, degree in top_cited:
        print(f"Citations: {degree}, Title: {internal_graph.nodes[node].get('title', 'N/A')}")

In [16]:
sch = SemanticScholar()

list_of_paper_ids = ["10.1111/gcb.12869"]
print(list_of_paper_ids)
fields = [
    'paperId', 'title', 'externalIds',
    'references.paperId', 'references.externalIds',
    'citations.paperId', 'citations.externalIds'
]

results = sch.get_paper(list_of_paper_ids[0], fields=fields) # 500 papers per query is maximum -> 400 queries for all data

print(results)

['10.1111/gcb.12869']


TypeError: 'NoneType' object is not iterable

In [21]:

# --- The "Magic" Function: Robust Batching with Recursive Fallback ---
def get_papers_robust(sch, doi_list, fields, failed_dois_list):
    """
    Fetches papers in a batch, with a recursive fallback for handling TypeErrors.

    Args:
        sch (SemanticScholar): The SemanticScholar client instance.
        doi_list (list): A list of DOIs to fetch.
        fields (list): The fields to request from the API.
        failed_dois_list (list): A list to append failed DOIs to.

    Returns:
        list: A list of successfully fetched Paper objects.
    """
    if not doi_list:
        return []

    try:
        # Attempt to fetch the entire batch
        return sch.get_papers(doi_list, fields=fields)
    except TypeError:
        # The batch failed, likely due to the NoneType bug.
        # If it's a single item, it's the culprit.
        if len(doi_list) == 1:
            failed_dois_list.append((doi_list[0], 'TypeError'))
            return []
        
        # If the batch was larger, divide and conquer.
        # tqdm.write just prints without messing up the progress bar
        tqdm.write(f"Batch of {len(doi_list)} failed. Splitting and retrying...")
        mid = len(doi_list) // 2
        first_half = get_papers_robust(sch, doi_list[:mid], fields, failed_dois_list)
        second_half = get_papers_robust(sch, doi_list[mid:], fields, failed_dois_list)
        
        return first_half + second_half
    except Exception as e:
        # Handle other potential errors (e.g., network issues) for the whole batch
        for doi in doi_list:
            failed_dois_list.append((doi, str(e)))
        return []

sch = SemanticScholar()



all_dois_in_corpus = [preprocess_doi(doi) for doi in df["DOI"].tolist()]
corpus_doi_set = set([doi for doi in all_dois_in_corpus if doi])
all_dois_to_query = list(corpus_doi_set)[::1000] # Query unique DOIs

print(f"Created a lookup set with {len(corpus_doi_set)} unique DOIs from your corpus.")

# --- API Workflow Using the Robust Batch Function ---
fields_to_request = ['externalIds', 'citations.externalIds']
chunk_size = 500  # API limit
doi_chunks = [all_dois_to_query[i:i + chunk_size] for i in range(0, len(all_dois_to_query), chunk_size)]

all_s2_papers = []
failed_dois = []

print(f"Querying Semantic Scholar in {len(doi_chunks)} large batches...")
for chunk in tqdm(doi_chunks, desc="Fetching paper data"):
    # Call our new robust function on each chunk
    results = get_papers_robust(sch, chunk, fields_to_request, failed_dois)
    all_s2_papers.extend(results)
    time.sleep(3) # A 3-second sleep between large batches is good practice

print(f"\nSuccessfully processed API calls.")
print(f"Fetched data for {len(all_s2_papers)} papers.")
print(f"Identified {len(failed_dois)} DOIs that could not be processed.")

# --- Graph Construction and Analysis (Same as before) ---
internal_links = []
for paper_data in tqdm(all_s2_papers, desc="Building Links"):
    if paper_data and paper_data.externalIds and paper_data.externalIds.get('DOI'):
        cited_paper_doi = paper_data.externalIds['DOI']
        if paper_data.citations:
            for citing_paper in paper_data.citations:
                if citing_paper and citing_paper.externalIds and citing_paper.externalIds.get('DOI'):
                    citing_paper_doi = citing_paper.externalIds['DOI']
                    if citing_paper_doi in corpus_doi_set:
                        internal_links.append({'source_doi': citing_paper_doi, 'target_doi': cited_paper_doi})

internal_links_df = pd.DataFrame(internal_links).drop_duplicates()
print(f"\nFound {len(internal_links_df)} unique internal links.")

G = nx.from_pandas_edgelist(internal_links_df, source='source_doi', target='target_doi', create_using=nx.DiGraph())
print(f"Graph created with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

https
Created a lookup set with 167330 unique DOIs from your corpus.
Querying Semantic Scholar in 1 large batches...


Fetching paper data:   0%|          | 0/1 [00:00<?, ?it/s]

/home/p0l3/.local/lib/python3.10/site-packages/semanticscholar/AsyncSemanticScholar.py:193: UserWarning: IDs not found: ['/10.3390/w15050866', '/10.3390/f13071109', '/10.3390/en10111780', '/10.3390/w12061658', '/10.3390/en10081121', '/10.3390/en11102832', '/10.3390/w12061539', '/10.3390/en11092360', '/10.3390/en10050699', '/10.3390/en10122172', '/10.3390/atmos13081305', '/10.3390/f14051059', '/10.3390/w15010206', '/10.3390/w10010036', '/10.3390/atmos3010124', '/10.3390/f14061273', '/10.3390/w13192679', '/10.3390/f8040133', '/10.3390/w15193365', '/10.3390/atmos13071153', '/10.3390/w10081011', '/10.3390/atmos13091492', '/10.3390/f10020102', '/10.3390/f12091206', '/10.3390/f12070913', '/10.3390/cli5040094', '/10.3390/f8060208', '/10.3390/atmos11060550', '/10.3390/en4122249', '/10.3390/en10121984', '/10.3390/en10091245', '/10.3390/en7128554', '/10.3390/atmos13030464', '/10.3390/en9060464', '/10.3390/atmos11091009', '/10.3390/w14131993', '/10.3390/w12040930', '/10.3390/atmos13071147', '/10.


Successfully processed API calls.
Fetched data for 114 papers.
Identified 0 DOIs that could not be processed.


Building Links:   0%|          | 0/114 [00:00<?, ?it/s]


Found 201 unique internal links.
Graph created with 232 nodes and 201 edges.


In [18]:
all_s2_data

[{'paperId': '79512924361554e34da2b9814746074a8b8973a9',
  'externalIds': {'MAG': '1963081755',
   'DOI': '10.1002/joc.3724',
   'CorpusId': 16315115},
  'title': 'A synoptic climatology of the near‐surface wind along the west coast of South America',
  'openAccessPdf': {'url': '',
   'status': 'CLOSED',
   'license': None,
   'disclaimer': "Notice: The following paper fields have been elided by the publisher: {'references'}. Paper or abstract available at https://api.unpaywall.org/v2/10.1002/joc.3724?email=<INSERT_YOUR_EMAIL> or https://doi.org/10.1002/joc.3724, which is subject to the license by the author or copyright owner provided with this content. Please go to the source to verify the license and copyright information for your use."},
  'authors': [{'authorId': '3375913', 'name': 'D. Rahn'},
   {'authorId': '11345182', 'name': 'R. Garreaud'}],
  'citations': [{'paperId': '8de597fbc7c46ff0bdfba7c62faadc9225d91788',
    'externalIds': {'DOI': '10.1029/2024jc021444', 'CorpusId': 27